In [ ]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage
import pandas as pd 

# --- TERMINAL UI COLORS ---
C = {'G': '\033[92m', 'R': '\033[91m', 'Y': '\033[93m', 'B': '\033[94m', 'C': '\033[96m', 'W': '\033[0m'}

# --- PARAMETERS ---
STOP_LOSS_PCT = -0.01 
TRAILING_STOP_PCT = -0.015  

# --- FUNCTIONS ---
def log_system_event(message):
    timestamp = time.ctime()
    with open('system_log.txt', 'a') as f:
        f.write(f'[{timestamp}], {message}\n')
    
def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\nAction: {signal}\nPrice: {price:.2f}\nPnL: {pnl:.2f}')
    msg['Subject'] = f'Bot Alert: {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login('smtp.quant@gmail.com', 'My Passcode')
        smtp.send_message(msg)

# --- INITIALIZE MEMORY & SCOREBOARD ---
last_buy_price = 0
last_signal = 'None'
highest_price = 0
total_pnl = 0.0
wins, losses = 0, 0

if os.path.exists('trade_log.csv'):
    try:
        df_history = pd.read_csv('trade_log.csv', names=['Time', 'Price', 'Signal', 'PnL'])
        df_history['PnL'] = pd.to_numeric(df_history['PnL'], errors='coerce').fillna(0)
        total_pnl = df_history['PnL'].sum()
        wins = len(df_history[df_history['PnL'] > 0])
        losses = len(df_history[df_history['PnL'] < 0])
        last_signal = str(df_history.iloc[-1]['Signal']).strip()
        if last_signal == 'BUY':
            last_buy_price = float(df_history.iloc[-1]['Price'])
            highest_price = last_buy_price
    except: pass

# ==========================================
# DAY 29: INITIAL DATA WARM-UP (PRE-LOOP)
# ==========================================
print(f"{C['B']}⏳ Warming up engines: Fetching historical baseline...{C['W']}")
master_df = yf.download('RELIANCE.NS', period='2d', interval='1m', progress=False)
print(f"{C['G']}🚀 Quantum Engine: Online (Incremental Stream Active){C['W']}\n")

while True:
    try:
        # --- DAY 29 REVISED FIX ---
        # We ask for '1d' (today's data) but in '1m' chunks
        new_data = yf.download('RELIANCE.NS', period='1d', interval='1m', progress=False)
        
        if not new_data.empty:
            # We only want the very last 1-minute candle to 'stitch'
            new_candle = new_data.tail(1) 
            
            # Stitch it to our Master DataFrame
            master_df = pd.concat([master_df, new_candle])
            
            # Clean: Remove any duplicate timestamps and keep the buffer at 250
            master_df = master_df[~master_df.index.duplicated(keep='last')]
            master_df = master_df.tail(250)
        else:
            print(f"\n{C['Y']}⚠️ Data gap detected. Waiting...{C['W']}")
            time.sleep(10)
            continue
            

        # --- TECHNICAL INDICATORS (Running on master_df) ---
        master_df['SMA_45'] = master_df['Close']['RELIANCE.NS'].rolling(window=45).mean()
        master_df['SMA_190'] = master_df['Close']['RELIANCE.NS'].rolling(window=190).mean()
        
        # RSI Calculation (Sliding Window)
        delta = master_df['Close']['RELIANCE.NS'].diff()
        gain = delta.clip(lower=0)
        loss = delta.clip(upper=0).abs()
        avg_gain = gain.rolling(window=14).mean()
        avg_loss = loss.rolling(window=14).mean()
        rs = avg_gain / avg_loss
        master_df['RSI'] = 100 - (100 / (1 + rs))
        
        latest = master_df.iloc[-1]
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()
        sma_45 = latest['SMA_45'].item()
        sma_190 = latest['SMA_190'].item()

        # --- RISK MANAGEMENT (Trailing Stop) ---
        if last_signal == 'BUY' and last_buy_price > 0:
            current_pnl_pct = (current_price - last_buy_price) / last_buy_price 
            if current_price > highest_price: highest_price = current_price
            drawdown = (current_price - highest_price) / highest_price
            
            trigger = None
            if current_pnl_pct <= STOP_LOSS_PCT: trigger = 'STOP_LOSS'
            elif drawdown <= TRAILING_STOP_PCT: trigger = 'TRAILING_STOP'
            
            if trigger:
                pnl = current_price - last_buy_price
                total_pnl += pnl
                if pnl > 0: wins += 1 
                else: losses += 1
                
                print(f"\n{C['R']}🚨 {trigger} HIT! Exit at {current_price:.2f}{C['W']}")
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger},{pnl}\n')
                
                try: send_trade_notification(trigger, current_price, pnl)
                except: pass
                
                last_signal, last_buy_price, highest_price = 'COOLDOWN', 0, 0
                time.sleep(60)
                continue

        # --- STRATEGY & EXECUTION ---
        current_signal = 'BUY' if (sma_45 > sma_190 and current_rsi < 60) else 'SELL'

        if last_signal == 'COOLDOWN':
            if current_signal == 'SELL': last_signal = 'SELL'
        elif current_signal != last_signal:
            pnl = 0
            if current_signal == 'SELL' and last_buy_price > 0:
                pnl = current_price - last_buy_price
                total_pnl += pnl
                if pnl > 0: wins += 1 
                else: losses += 1
                print(f"\n{C['Y']}💰 Trend Exit at {current_price:.2f}{C['W']}")
            elif current_signal == 'BUY':
                last_buy_price = highest_price = current_price
                print(f"\n{C['G']}📥 Entry at {current_price:.2f}{C['W']}")
            
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
            try: send_trade_notification(current_signal, current_price, pnl)
            except: pass
            last_signal = current_signal

        # --- LIVE DASHBOARD TICKER ---
        pnl_val = current_price - last_buy_price if last_signal == 'BUY' else 0
        p_col = C['G'] if pnl_val >= 0 else C['R']
        t_col = C['G'] if total_pnl >= 0 else C['R']
        
        print(f"\r{C['B']}[{time.strftime('%H:%M:%S')}] {C['W']}"
              f"Prc: {current_price:.2f} | RSI: {C['C']}{current_rsi:.1f}{C['W']} | "
              f"Trade: {p_col}{pnl_val:+.2f}{C['W']} | Total: {t_col}{total_pnl:+.2f}{C['W']} | "
              f"W/L: {wins}/{losses} | {C['Y']}Buff: {len(master_df)}m{C['W']}    ", end="")

    except Exception as e:
        print(f"\n{C['R']}⚠️ Error: {e}{C['W']}")
        log_system_event(str(e))
        time.sleep(15)

    time.sleep(60)

⏳ Warming up engines: Fetching historical baseline...
🚀 Quantum Engine: Online (Incremental Stream Active)

[15:12:45] Prc: 1357.50 | RSI: 28.7 | Trade: +0.00 | Total: +0.00 | W/L: 0/0 | Buff: 250m    